# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their fields/columns by @id
print('Available RecordSets:')
record_sets = []

# metadata.record_sets is a list of RecordSet objects with `@id` and `fields`/`columns` attributes
for rs in metadata.record_sets:
    print(f"  RecordSet @id: {rs.id}")
    record_sets.append(rs.id)
    print("    Fields/Columns by @id:")
    for fld in getattr(rs, 'fields', []) + getattr(rs, 'columns', []):
        print(f"     - {fld.id} ({fld.name})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets using their @id
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records from RecordSet: {record_set_id}")
    try:
        recs = list(dataset.records(record_set=record_set_id))
    except Exception as e:
        print(f"  Failed to load: {e}")
        continue
    df = pd.DataFrame(recs)
    dataframes[record_set_id] = df
    if not df.empty:
        print(f"    Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print("    (No records found)")
    print("-")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA on the main proteins table
import numpy as np
warnings.filterwarnings('ignore')

# For example purposes, we'll select the first non-empty record set
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        break

if main_record_set_id is None:
    print("No non-empty record set found for analysis.")
else:
    df = dataframes[main_record_set_id]
    print(f"Using record set: {main_record_set_id}")
    print("Columns detected:", df.columns.tolist())

    # Try to automatically pick a numeric field
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_candidates) == 0:
        # Try to convert columns to numeric where possible
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c])
            except Exception:
                continue
        numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()

    if len(numeric_candidates) > 0:
        numeric_field = numeric_candidates[0]
        print(f"Numeric field chosen for filtering and normalization: {numeric_field}")
        threshold = np.percentile(df[numeric_field].dropna(), 75)  # use 75th percentile as threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records")
        print(filtered_df.head())

        norm_field = f"{numeric_field}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() or 1e-9)
        print(filtered_df[[numeric_field, norm_field]].head())

        # Try grouping by a string/categorical field
        group_fields = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            group_field = None
            print('No suitable group/categorical field detected.')
    else:
        print("No numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and len(numeric_candidates) > 0:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field}")
    plt.show()

    if group_field is not None:
        # Plot boxplot grouped
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, process, and visualize a dataset described by a Croissant schema using the `mlcroissant` library. By referencing all data structures by their `@id`, users can ensure reproducible and robust data access. Further domain-specific analysis can be performed as needed to advance biological or proteomics insights.